# Signal Processing Unit (SPU) Demo

This notebook rebuilds the project in a clean, working form using the audio clips already present in this folder.

Pipeline:
1. Load `.wav` signals.
2. Inspect waveform and spectrogram.
3. Apply low-pass filtering.
4. Compute FFT.
5. Extract DSP and MFCC features.
6. Train a simple classifier on fixed-length signal segments.
7. Predict the source clip of an unseen segment.

Note: the folder does not include human-annotated class labels, so the ML stage uses the source audio file name as the label. This keeps the pipeline executable while still demonstrating feature-based audio classification.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
from scipy import signal as spsignal
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

plt.rcParams['figure.figsize'] = (12, 4)

DATA_DIR = Path('.')
AUDIO_FILES = sorted(DATA_DIR.glob('sample*.wav'))
if not AUDIO_FILES:
    raise FileNotFoundError('No sample*.wav files were found in the current folder.')

SEGMENT_SECONDS = 0.5
LOWPASS_CUTOFF_HZ = 4000
RANDOM_STATE = 42

print('Audio files found:')
for path in AUDIO_FILES:
    print('-', path.name)

In [ ]:
audio_bank = {}
metadata_rows = []

for path in AUDIO_FILES:
    audio, sr = librosa.load(path, sr=None, mono=True)
    audio_bank[path.stem] = {'path': path, 'audio': audio, 'sr': sr}
    metadata_rows.append(
        {
            'file': path.name,
            'sample_rate_hz': sr,
            'samples': len(audio),
            'duration_seconds': len(audio) / sr,
            'peak_amplitude': float(np.max(np.abs(audio))),
            'mean_amplitude': float(np.mean(np.abs(audio))),
        }
    )

metadata_df = pd.DataFrame(metadata_rows).sort_values('duration_seconds').reset_index(drop=True)
metadata_df

In [ ]:
example_key = sorted(audio_bank.keys(), key=lambda key: len(audio_bank[key]['audio']))[0]
example_audio = audio_bank[example_key]['audio']
example_sr = audio_bank[example_key]['sr']

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
librosa.display.waveshow(example_audio, sr=example_sr, ax=axes[0], color='teal')
axes[0].set_title(f'Waveform: {example_key}')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')

spectrogram_db = librosa.amplitude_to_db(np.abs(librosa.stft(example_audio)), ref=np.max)
img = librosa.display.specshow(spectrogram_db, sr=example_sr, x_axis='time', y_axis='hz', ax=axes[1], cmap='magma')
axes[1].set_title(f'Spectrogram: {example_key}')
fig.colorbar(img, ax=axes[1], format='%+2.0f dB')
plt.tight_layout()
plt.show()

In [ ]:
nyquist = 0.5 * example_sr
normalized_cutoff = LOWPASS_CUTOFF_HZ / nyquist
b, a = spsignal.butter(4, normalized_cutoff, btype='low')
filtered_audio = spsignal.filtfilt(b, a, example_audio)

segment_length = min(example_sr * 2, len(example_audio))
raw_segment = example_audio[:segment_length]
filtered_segment = filtered_audio[:segment_length]
freq_axis = np.fft.rfftfreq(segment_length, d=1 / example_sr)
raw_fft = np.abs(np.fft.rfft(raw_segment))
filtered_fft = np.abs(np.fft.rfft(filtered_segment))

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
axes[0].plot(raw_segment, label='Raw', alpha=0.8)
axes[0].plot(filtered_segment, label='Filtered', alpha=0.8)
axes[0].set_title('Time-Domain Comparison')
axes[0].set_xlabel('Sample Index')
axes[0].set_ylabel('Amplitude')
axes[0].legend()

axes[1].plot(freq_axis, raw_fft, label='Raw FFT', alpha=0.7)
axes[1].plot(freq_axis, filtered_fft, label='Filtered FFT', alpha=0.9)
axes[1].set_title('Frequency-Domain Comparison')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Magnitude')
axes[1].set_xlim(0, min(8000, example_sr / 2))
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
def extract_features(segment: np.ndarray, sr: int) -> dict:
    feature_dict = {
        'rms_mean': float(np.mean(librosa.feature.rms(y=segment))),
        'zcr_mean': float(np.mean(librosa.feature.zero_crossing_rate(segment))),
        'spectral_centroid_mean': float(np.mean(librosa.feature.spectral_centroid(y=segment, sr=sr))),
        'spectral_bandwidth_mean': float(np.mean(librosa.feature.spectral_bandwidth(y=segment, sr=sr))),
        'spectral_rolloff_mean': float(np.mean(librosa.feature.spectral_rolloff(y=segment, sr=sr))),
        'spectral_flatness_mean': float(np.mean(librosa.feature.spectral_flatness(y=segment))),
    }

    mfcc = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=13)
    for index, row in enumerate(mfcc, start=1):
        feature_dict[f'mfcc_{index}_mean'] = float(np.mean(row))
        feature_dict[f'mfcc_{index}_std'] = float(np.std(row))

    return feature_dict


segment_counts = []
for key, item in audio_bank.items():
    samples_per_segment = int(item['sr'] * SEGMENT_SECONDS)
    usable_segments = len(item['audio']) // samples_per_segment
    segment_counts.append(usable_segments)

segments_per_class = min(segment_counts)
rows = []

for key, item in audio_bank.items():
    audio = item['audio']
    sr = item['sr']
    samples_per_segment = int(sr * SEGMENT_SECONDS)

    for segment_index in range(segments_per_class):
        start = segment_index * samples_per_segment
        stop = start + samples_per_segment
        segment = audio[start:stop]
        row = extract_features(segment, sr)
        row['label'] = key
        row['file'] = item['path'].name
        row['segment_index'] = segment_index
        rows.append(row)

feature_df = pd.DataFrame(rows)
print(f'Balanced segments per class: {segments_per_class}')
print(f'Dataset shape: {feature_df.shape}')
feature_df.head()

In [ ]:
X = feature_df.drop(columns=['label', 'file', 'segment_index'])
y = feature_df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

model = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    class_weight='balanced',
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Test accuracy: {accuracy:.3f}')
print('\nClassification report:')
print(classification_report(y_test, y_pred))

labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap='Blues')
plt.title('Confusion Matrix')
plt.xticks(range(len(labels)), labels, rotation=45)
plt.yticks(range(len(labels)), labels)
plt.xlabel('Predicted')
plt.ylabel('Actual')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha='center', va='center', color='black')
plt.tight_layout()
plt.show()

In [ ]:
demo_row = feature_df.sample(1, random_state=RANDOM_STATE).iloc[0]
demo_features = demo_row.drop(labels=['label', 'file', 'segment_index']).to_frame().T
demo_prediction = model.predict(demo_features)[0]
demo_probabilities = pd.Series(model.predict_proba(demo_features)[0], index=model.classes_).sort_values(ascending=False)

print('Demo prediction')
print('Actual label   :', demo_row['label'])
print('Predicted label:', demo_prediction)
print('Source file    :', demo_row['file'])
print('Segment index  :', int(demo_row['segment_index']))
print('\nClass probabilities:')
demo_probabilities